# 14 — Cache and Persist Pattern

Purpose: reuse the same DataFrame multiple times without recomputing the full lineage again.

Process schema:

```text
SOURCE DATA
  |
  v
EXPENSIVE TRANSFORMATION
  |
  v
CACHE / PERSIST
  |
  +--> ACTION 1
  +--> ACTION 2
  +--> ACTION 3
```

Use this pattern when the same transformed DataFrame is reused multiple times.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("cache-persist-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

## Step 1 — Input data

In [ ]:
df = spark.createDataFrame(
    [
        ("e1", "u1", "purchase", 10.0),
        ("e2", "u1", "purchase", 20.0),
        ("e3", "u2", "refund", 5.0),
        ("e4", "u2", "purchase", 30.0),
        ("e5", "u3", "purchase", 40.0),
    ],
    ["event_id", "user_id", "event_type", "amount"]
)

df.show(truncate=False)

## Step 2 — Expensive transformation

In a real pipeline this could be:
- joins
- parsing
- normalization
- validation
- aggregations

In [ ]:
prepared = (
    df
    .withColumn("amount_with_tax", F.col("amount") * F.lit(1.2))
    .withColumn("is_purchase", F.col("event_type") == F.lit("purchase"))
)

prepared.show(truncate=False)

## Step 3 — Cache when reused

`cache()` is shorthand for memory/disk storage.

Important:

```text
cache is lazy
```

It is populated only after the first action.

In [ ]:
cached_prepared = prepared.cache()

cached_prepared.count()

## Step 4 — Reuse cached DataFrame

Now multiple actions can reuse the cached result.

In [ ]:
by_user = (
    cached_prepared
    .groupBy("user_id")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
        F.sum("amount_with_tax").alias("total_amount_with_tax"),
    )
)

purchase_only = cached_prepared.filter(F.col("is_purchase"))

by_user.show(truncate=False)
purchase_only.show(truncate=False)

## Step 5 — Persist with explicit storage level

Use `persist(...)` when you want to choose storage behavior explicitly.

In [ ]:
persisted_prepared = prepared.persist(StorageLevel.MEMORY_AND_DISK)

persisted_prepared.count()
persisted_prepared.storageLevel

## Step 6 — Unpersist when finished

Free executor memory after the reused DataFrame is no longer needed.

In [ ]:
cached_prepared.unpersist()
persisted_prepared.unpersist()

print("unpersisted")

## Step 7 — Control totals

In [ ]:
control_totals = spark.createDataFrame(
    [
        ("input_rows", df.count()),
        ("prepared_rows", prepared.count()),
        ("purchase_rows", purchase_only.count()),
    ],
    ["metric", "value"]
)

control_totals.show(truncate=False)

## Final result

```text
PREPARED DATAFRAME
  |
  v
CACHE / PERSIST
  |
  +--> reused by multiple outputs
  |
  v
UNPERSIST
```

This pattern is useful when the same DataFrame is consumed more than once.

In [ ]:
spark.stop()